# 02 - Bandit Training

This notebook reviews the saved simulation output across all policies, focusing on how revenue, defaults, and action choices evolve over the 12-month run.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


## Section 1: Load Simulation Results and Verify Policies

We first load the saved policy outcomes and confirm that all expected policies are present. The action distribution table shows how aggressively each policy changed customer limits over the simulation.

In [ ]:
results_path = Path('../data/simulation_results.csv')
if not results_path.exists():
    raise FileNotFoundError('Run src/simulate_run.py first to generate data/simulation_results.csv')

results_df = pd.read_csv(results_path)
policy_alias = {
    'thompson_sampling': 'thompson',
    'ucb': 'ucb',
    'epsilon_greedy': 'epsilon_greedy',
    'static_baseline': 'static',
    'oracle': 'oracle',
}
results_df['policy_display'] = results_df['policy'].map(policy_alias).fillna(results_df['policy'])

expected_policies = {'thompson', 'ucb', 'epsilon_greedy', 'static', 'oracle'}
present_policies = set(results_df['policy_display'].unique())
assert expected_policies.issubset(present_policies), f'Missing policies: {expected_policies - present_policies}'
print('Policies present:', sorted(present_policies))
display(results_df.head())

action_distribution = (
    results_df.groupby(['policy_display', 'action_taken'])
    .size()
    .groupby(level=0)
    .apply(lambda s: s / s.sum() * 100.0)
    .rename('pct_of_decisions')
    .reset_index()
)
action_distribution_pivot = action_distribution.pivot(index='policy_display', columns='action_taken', values='pct_of_decisions').fillna(0.0)
display(action_distribution_pivot.style.format('{:.2f}'))


## Section 2: Learning Curves

Cumulative reward is the core training outcome. This chart compares how quickly each policy compounds revenue and how the portfolio trajectory changes around the month-6 shock event.

In [ ]:
monthly_reward = (
    results_df.groupby(['policy_display', 'month'], as_index=False)['reward_received']
    .sum()
    .sort_values(['policy_display', 'month'])
)
monthly_reward['cumulative_reward'] = monthly_reward.groupby('policy_display')['reward_received'].cumsum()

fig_learning = px.line(
    monthly_reward,
    x='month',
    y='cumulative_reward',
    color='policy_display',
    markers=True,
    title='Cumulative Portfolio Revenue by Policy (12-month simulation)',
)
fig_learning.add_vline(x=6, line_dash='dash', line_color='firebrick', annotation_text='Shock', annotation_position='top')
fig_learning.update_layout(xaxis_title='Month', yaxis_title='Cumulative Reward (INR)')
fig_learning.show()


## Section 3: Monthly Default Rate

Revenue only matters if default risk stays under control. This chart isolates monthly default behavior for the three learning policies plus the static benchmark so we can see the shock spike and any subsequent recovery.

In [ ]:
default_policies = ['thompson', 'ucb', 'epsilon_greedy', 'static']
monthly_default = (
    results_df.loc[results_df['policy_display'].isin(default_policies)]
    .groupby(['policy_display', 'month'], as_index=False)['did_default']
    .mean()
)
monthly_default['default_rate_pct'] = monthly_default['did_default'] * 100.0

fig_default = px.line(
    monthly_default,
    x='month',
    y='default_rate_pct',
    color='policy_display',
    markers=True,
    title='Monthly Default Rate by Policy',
)
fig_default.add_vline(x=6, line_dash='dash', line_color='firebrick', annotation_text='Shock', annotation_position='top')
fig_default.add_vrect(x0=6, x1=8, line_width=0, fillcolor='firebrick', opacity=0.08)
fig_default.update_layout(xaxis_title='Month', yaxis_title='Default Rate (%)')
fig_default.show()


## Section 4: Action Distribution Evolution

This stacked view focuses on Thompson Sampling and shows how the policy allocates decisions across keep, +10, +20, and +50 actions over time. It is a direct read on how exploration and exploitation evolve month by month.

In [ ]:
thompson_actions = results_df.loc[results_df['policy_display'] == 'thompson']
action_evolution = (
    thompson_actions.groupby(['month', 'action_taken'])
    .size()
    .groupby(level=0)
    .apply(lambda s: s / s.sum() * 100.0)
    .rename('pct_of_actions')
    .reset_index()
)

fig_actions = px.bar(
    action_evolution,
    x='month',
    y='pct_of_actions',
    color='action_taken',
    title='Thompson Sampling Action Mix Over Time',
)
fig_actions.update_layout(barmode='stack', xaxis_title='Month', yaxis_title='Action Share (%)')
fig_actions.show()


## Section 5: Revenue Lift Summary

The final summary compares total 12-month revenue across policies and annotates every bar relative to the static baseline. This makes it easy to see whether each learning strategy produced meaningful business lift.

In [ ]:
policy_revenue = (
    results_df.groupby('policy_display', as_index=False)['reward_received']
    .sum()
    .rename(columns={'reward_received': 'total_revenue'})
)
static_revenue = float(policy_revenue.loc[policy_revenue['policy_display'] == 'static', 'total_revenue'].iloc[0])
policy_revenue['lift_vs_static_pct'] = np.where(
    static_revenue != 0,
    (policy_revenue['total_revenue'] - static_revenue) / static_revenue * 100.0,
    0.0,
)
policy_revenue = policy_revenue.sort_values('total_revenue', ascending=False)

fig_revenue = px.bar(
    policy_revenue,
    x='policy_display',
    y='total_revenue',
    color='policy_display',
    text=policy_revenue['lift_vs_static_pct'].map(lambda v: f'{v:+.1f}%'),
    title='Total 12-month Revenue by Policy',
)
fig_revenue.add_hline(y=static_revenue, line_dash='dash', line_color='gray', annotation_text='Static baseline')
fig_revenue.update_layout(showlegend=False, xaxis_title='Policy', yaxis_title='Total Revenue (INR)')
fig_revenue.update_traces(textposition='outside')
fig_revenue.show()
